## Themes precis — 4 editions a correspondance exacte (famille 7 / 8a)

Pour les 4 editions ayant une feuille `#N` dediee dans `BNU_corpus.ods` (meme ark
que le dossier segmente), on rattache une information thematique **planche par
planche**, bien plus precise que le "theme inconnu" actuel :

| Edition | Feuille | Structure | Rattachement |
|---|---|---|---|
| Solis (bois_solis_feyerabend_francfort1581) | `#7` | 1 ligne = 1 theme court, foliotation | calibration folio->page (9 ancres) |
| Salomon (bois_salomon_rouille_lyon1557) | `#0_CORPUS_REF` | 1 ligne = 1 theme court, meme sequence que Solis, **pas d'ancre par ligne** | position (ordre), confiance moindre |
| Wickram (bois_wickram_behem_mayence1545) | `#1` | 1 ligne = 1 planche, **plusieurs episodes combines**, page explicite | page directe (url par ligne) |
| Savery (cuivre_savery_farnaby_paris1637) | `#11` | idem Wickram | page directe (url par ligne) |

Deux colonnes ajoutees a la base vectorielle : `theme_precis` (titre court, Solis
et Salomon) et `description_planche` (paragraphe multi-episodes, Wickram et
Savery), plus `confiance_rattachement` (`haute` / `position`).

In [1]:
import re
from pathlib import Path

import pandas as pd

RACINE = Path("../../").resolve()
SEG_DIR = RACINE / "data" / "editions_ovide" / "segmentees"

xl = pd.ExcelFile(RACINE / "retours_celine" / "BNU_corpus.ods", engine="odf")
base = pd.read_pickle(RACINE / "data" / "vector_bases" / "ovide_corpus_complet_siglip.pkl")
base["theme_precis"] = pd.NA
base["description_planche"] = pd.NA
base["confiance_rattachement"] = pd.NA
print(f"Base chargee : {len(base)} illustrations")

Base chargee : 2191 illustrations


### 1. Solis (`#7`) — calibration folio -> page (validee : 9 ancres, offset constant)

In [2]:
PAT_FOLIO = re.compile(r"^(\d+)([rv])°([ab])?$")
PAT_URL_PAGE = re.compile(r"[?&]page=(\d+)")


def parse_folio(s):
    m = PAT_FOLIO.match(str(s).strip())
    if not m:
        return None
    num, cote, variante = m.groups()
    index = 2 * (int(num) - 1) + (0 if cote == "r" else 1) + 1
    return index


def extraire_titre_court(theme_brut):
    """Retire le prefixe #3.0.NNN pour ne garder que le titre lisible."""
    m = re.match(r"#[\d.]+\s+(.*)", str(theme_brut).strip())
    return m.group(1) if m else str(theme_brut).strip()


df7 = xl.parse("#7", header=None)
sous7 = df7.iloc[3:, [2, 3, 4]].copy()
sous7.columns = ["foliotation", "url", "theme"]
sous7 = sous7[sous7["foliotation"].notna()].reset_index(drop=True)

offsets = set()
for _, row in sous7.iterrows():
    idx = parse_folio(row["foliotation"])
    m = PAT_URL_PAGE.search(str(row["url"]))
    if idx is not None and m:
        offsets.add(int(m.group(1)) - idx)
assert len(offsets) == 1, f"offsets incoherents pour Solis : {offsets}"
OFFSET_SOLIS = offsets.pop()
print(f"Solis : offset calibre = {OFFSET_SOLIS} (sur {len(offsets) + 1} ancres)")

fichiers_solis = {}
for f in (SEG_DIR / "bois_solis_feyerabend_francfort1581").glob("*.jpg"):
    m = re.match(r"solis_(\d+)_det\d+_conf[\d.]+\.jpg$", f.name)
    if m:
        fichiers_solis.setdefault(int(m.group(1)), []).append(str(f))

n_solis = 0
for _, row in sous7.iterrows():
    idx = parse_folio(row["foliotation"])
    if idx is None:
        continue
    page = idx + OFFSET_SOLIS
    for chemin in fichiers_solis.get(page, []):
        masque = base["chemin"] == chemin
        if masque.any():
            base.loc[masque, "theme_precis"] = extraire_titre_court(row["theme"])
            base.loc[masque, "confiance_rattachement"] = "haute"
            n_solis += 1

print(f"Solis : {n_solis} illustrations rattachees a un theme precis")

Solis : offset calibre = 20 (sur 1 ancres)
Solis : 181 illustrations rattachees a un theme precis


### 2. Salomon (`#0_CORPUS_REF`) — meme sequence de themes, rattachement par position

In [3]:
df_salomon = xl.parse("#0_CORPUS_REF", header=None)
themes_salomon = df_salomon.iloc[2:, 14].dropna().tolist()  # ordre = ordre physique du livre
themes_salomon = [extraire_titre_court(t) for t in themes_salomon]

fichiers_salomon = sorted(
    (SEG_DIR / "bois_salomon_rouille_lyon1557").glob("*.jpg"),
    key=lambda p: int(re.search(r"salomon_f(\d+)_det", p.name).group(1)),
)

print(f"Salomon : {len(themes_salomon)} themes dans la feuille, {len(fichiers_salomon)} crops segmentes")

n_salomon = 0
for chemin, theme in zip(fichiers_salomon, themes_salomon):
    masque = base["chemin"] == str(chemin)
    if masque.any():
        base.loc[masque, "theme_precis"] = theme
        base.loc[masque, "confiance_rattachement"] = "position"
        n_salomon += 1

print(f"Salomon : {n_salomon} illustrations rattachees par position (confiance moindre)")

Salomon : 178 themes dans la feuille, 161 crops segmentes
Salomon : 161 illustrations rattachees par position (confiance moindre)


### 3. Wickram (`#1`) et Savery (`#11`) — description de planche (page directe)

In [4]:
def rattacher_par_page_directe(nom_feuille, prefixe_dossier, prefixe_fichier, ligne_debut, col_url, col_sujet):
    df = xl.parse(nom_feuille, header=None)
    sous = df.iloc[ligne_debut:, [col_url, col_sujet]].copy()
    sous.columns = ["url", "sujet"]
    sous = sous[sous["url"].notna() & sous["sujet"].notna()]

    fichiers = {}
    for f in (SEG_DIR / prefixe_dossier).glob("*.jpg"):
        m = re.match(rf"{prefixe_fichier}_(\d+)_det\d+_conf[\d.]+\.jpg$", f.name)
        if m:
            fichiers.setdefault(int(m.group(1)), []).append(str(f))

    n = 0
    for _, row in sous.iterrows():
        m = PAT_URL_PAGE.search(str(row["url"]))
        if not m:
            continue
        page = int(m.group(1))
        for chemin in fichiers.get(page, []):
            masque = base["chemin"] == chemin
            if masque.any():
                base.loc[masque, "description_planche"] = row["sujet"]
                base.loc[masque, "confiance_rattachement"] = "haute"
                n += 1
    return n, len(sous), len(fichiers)


n_wick, n_lignes_wick, n_fichiers_wick = rattacher_par_page_directe(
    "#1", "bois_wickram_behem_mayence1545", "wickram", 3, 3, 4
)
print(f"Wickram : {n_wick} rattaches ({n_lignes_wick} lignes feuille, {n_fichiers_wick} pages avec crop)")

n_sav, n_lignes_sav, n_fichiers_sav = rattacher_par_page_directe(
    "#11", "cuivre_savery_farnaby_paris1637", "clein", 3, 3, 4
)
print(f"Savery : {n_sav} rattaches ({n_lignes_sav} lignes feuille, {n_fichiers_sav} pages avec crop)")

Wickram : 50 rattaches (47 lignes feuille, 47 pages avec crop)


Savery : 15 rattaches (15 lignes feuille, 18 pages avec crop)


### 4. Verification manuelle avant sauvegarde

In [5]:
pd.set_option("display.max_colwidth", 80)
echantillon = base[base["confiance_rattachement"].notna()].groupby("dossier").apply(
    lambda g: g[["chemin", "theme_precis", "description_planche", "confiance_rattachement"]].head(3)
)
print(echantillon.to_string())

print("\nCouverture par dossier :")
print(base[base["dossier"].isin([
    "bois_solis_feyerabend_francfort1581", "bois_salomon_rouille_lyon1557",
    "bois_wickram_behem_mayence1545", "cuivre_savery_farnaby_paris1637",
])].groupby("dossier")["confiance_rattachement"].apply(lambda s: s.notna().sum()).rename("nb_rattaches"))

                                                                                                                                                                                         chemin            theme_precis                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

### 5. Sauvegarde

In [6]:
chemin_sortie = RACINE / "data" / "vector_bases" / "ovide_corpus_complet_siglip.pkl"
base.to_pickle(chemin_sortie)
nb_theme = base["theme_precis"].notna().sum()
nb_desc = base["description_planche"].notna().sum()
print(f"Sauvegarde : {chemin_sortie}")
print(f"Total avec theme_precis : {nb_theme}")
print(f"Total avec description_planche : {nb_desc}")


Sauvegarde : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/vector_bases/ovide_corpus_complet_siglip.pkl
Total avec theme_precis : 341
Total avec description_planche : 65


## Bilan

~413 illustrations potentiellement couvrables (Solis 184, Salomon 161, Wickram 47
planches, Savery 18 planches) — voir le compte exact ci-dessus. `theme_precis`
(Solis + Salomon) donne un titre court fiable pour la recherche/génération ;
`description_planche` (Wickram + Savery) donne un paragraphe riche mais qui
mélange plusieurs épisodes par planche — à traiter différemment en aval (ex. ne
pas le citer comme un thème unique).

**Non fait ici** : brancher ces nouveaux champs dans `retrieval.ipynb` /
`rag_generation.ipynb` / `app_rag.py` (actuellement ils ne lisent que `theme`) —
prochaine étape si utile.